# Build the CheckAMG de-novo database

Package the best PST model checkpoint (selected in `train_pst.ipynb`) together with its reference
protein embeddings, FAISS index, and labels into a standalone, versioned CheckAMG de-novo database,
then write a README and tar it for distribution. This mirrors how the annotate database is built in
`build_annotate_db.ipynb`.

The de-novo database is a separate package from the annotate database and is downloaded into its own
directory by `checkamg download`. To cut a new version, bump `CHECKAMG_DENOVO_DB_VERSION` and
`CHECKAMG_DENOVO_DB_DATE` below.

## Setup

In [1]:
import os
from pathlib import Path

## Inputs from `train_pst.ipynb`

Set these to the best model selected by the model-selection steps in `train_pst.ipynb`. The
checkpoint is auto-discovered when there is a single checkpoint in the run; otherwise set
`BEST_MODEL_CKPT_NAME` explicitly to the one chosen there.

In [2]:
TRAIN_ROOT = Path("/storage2/scratch/kosmopoulos/projects/checkAMG/pst/training/train_outputs_chtc/")
RUN_ID = "checkamg_train_071426"
MODEL_OUTDIR = TRAIN_ROOT.joinpath(RUN_ID)

BEST_MODEL = "checkAMG-PST_TL-P__large_5" # best model outdir name from train_pst.ipynb
BEST_MODEL_CKPT_NAME = "" # set explicitly to override checkpoint auto-discovery

ckpt_dir = MODEL_OUTDIR / BEST_MODEL / "lightning_logs" / "version_0" / "checkpoints"
if BEST_MODEL_CKPT_NAME:
    BEST_MODEL_CKPT = ckpt_dir / BEST_MODEL_CKPT_NAME
else:
    # 'last.ckpt' is Lightning's final-epoch copy; the monitored best checkpoint has the metric in its name
    found = sorted(p for p in ckpt_dir.glob("*.ckpt") if p.name != "last.ckpt")
    assert len(found) == 1, (
        f"Expected exactly one monitored checkpoint in {ckpt_dir}; set BEST_MODEL_CKPT_NAME explicitly. "
        f"Found: {[f.name for f in found]}"
    )
    BEST_MODEL_CKPT = found[0]
assert BEST_MODEL_CKPT.exists(), f"Best model checkpoint not found at {BEST_MODEL_CKPT}"

BEST_MODEL_TRAIN_EMBED = MODEL_OUTDIR / BEST_MODEL / "checkAMG_train_esm2_t30_150M.graphfmt.PST-EMBED.h5"
assert BEST_MODEL_TRAIN_EMBED.exists(), f"Train embeddings not found at {BEST_MODEL_TRAIN_EMBED}"

print(f"Best model:       {BEST_MODEL}")
print(f"Checkpoint:       {BEST_MODEL_CKPT}")
print(f"Train embeddings: {BEST_MODEL_TRAIN_EMBED}")

Best model:       checkAMG-PST_TL-P__large_5
Checkpoint:       /storage2/scratch/kosmopoulos/projects/checkAMG/pst/training/train_outputs_chtc/checkamg_train_071426/checkAMG-PST_TL-P__large_5/lightning_logs/version_0/checkpoints/epoch=13_train_loss=0.114.ckpt
Train embeddings: /storage2/scratch/kosmopoulos/projects/checkAMG/pst/training/train_outputs_chtc/checkamg_train_071426/checkAMG-PST_TL-P__large_5/checkAMG_train_esm2_t30_150M.graphfmt.PST-EMBED.h5


## Export the best model checkpoint, FAISS index, and labels


In [3]:
import shutil
import subprocess
from dataclasses import dataclass


def runid_to_date_str(run_id: str) -> str:
    predate = run_id.split("_")[-1]
    if predate.startswith("20") and len(predate) == 8:
        return predate
    else:
        mm, dd, yy = predate[:2], predate[2:4], predate[4:]
        return f"20{yy}{mm}{dd}"


@dataclass
class DbEntry:
    component: str
    version: str
    accessed: str
    citation: str


def render_pretty_table(entries: list[DbEntry]) -> str:
    headers = ["component", "version", "accessed", "citation"]
    rows = [[e.component, e.version, e.accessed, e.citation] for e in entries]

    def esc(s: str) -> str:
        return str(s).replace("\t", " ").replace("\n", " ").strip()

    widths = [len(h) for h in headers]
    for r in rows:
        for i, cell in enumerate(r):
            widths[i] = max(widths[i], len(esc(cell)))

    header_line = "  ".join(headers[i].ljust(widths[i]) for i in range(len(headers)))
    sep_line = "-" * len(header_line)
    row_lines = [
        "  ".join(esc(r[i]).ljust(widths[i]) for i in range(len(headers)))
        for r in rows
    ]
    return "\n".join([header_line, sep_line] + row_lines)


def build_denovo_readme(
    db_version: str,
    db_date: str,
    file_prefix: str,
) -> str:
    return (
        f"CheckAMG de-novo database version {db_version} ({db_date})\n"
        "\n"
        "This is the reference database used by 'checkamg denovo'. It contains a trained Protein Set\n"
        "Transformer (PST) model and the precomputed reference protein embeddings, FAISS index, and\n"
        "labels derived from the training data. 'checkamg denovo' embeds query proteins with the model\n"
        "and assigns metabolic labels by distance-weighted k-nearest-neighbor search against the index.\n"
        "\n"
        "If you use CheckAMG de-novo for your research, please also cite the Protein Set Transformer:\n"
        "Martin et al. 2025 Nat. Commun. https://doi.org/10.1038/s41467-025-66049-4\n"
        "\n"
        "Contents:\n"
        f"  {file_prefix}.ckpt                   trained CheckAMG-PST model checkpoint\n"
        f"  {file_prefix}.PST-EMBED.h5           reference protein PST embeddings (with labels)\n"
        f"  {file_prefix}.PST-EMBED.index.faiss  FAISS index over the reference embeddings\n"
        f"  {file_prefix}.PST-EMBED.labels.h5    reference protein labels\n"
        "\n"
        "See https://github.com/AnantharamanLab/CheckAMG/blob/main/notebooks/build_denovo_db.ipynb for details.\n"
    )


def make_tar_gz_parallel(src_dir: str, threads: int = 0, out_path: str | None = None) -> Path:
    src = Path(src_dir).resolve()
    if not src.is_dir():
        raise FileNotFoundError(f"Source directory not found: {src}")

    if shutil.which("tar") is None:
        raise RuntimeError("tar not found on PATH")
    if shutil.which("pigz") is None:
        raise RuntimeError("pigz not found on PATH")

    out = Path(out_path).resolve() if out_path else src.parent / f"{src.name}.tar.gz"
    out.parent.mkdir(parents=True, exist_ok=True)

    cmd = ["tar", "-C", str(src.parent), "-cf", "-", src.name]
    pigz = ["pigz", "-9"]
    pigz += ["-p", str(threads)] if threads and threads > 0 else ["-p", "0"]

    with open(out, "wb") as f_out:
        p1 = subprocess.Popen(cmd, stdout=subprocess.PIPE)
        try:
            p2 = subprocess.Popen(pigz, stdin=p1.stdout, stdout=f_out)
        finally:
            if p1.stdout is not None:
                p1.stdout.close()
        rc2 = p2.wait()
        rc1 = p1.wait()

    if rc1 != 0:
        raise RuntimeError(f"tar failed with exit code {rc1}")
    if rc2 != 0:
        raise RuntimeError(f"pigz failed with exit code {rc2}")

    return out

In [4]:
CHECKAMG_DENOVO_DB_VERSION = 1.1
CHECKAMG_DENOVO_DB_DATE = runid_to_date_str(RUN_ID) # YYYYMMDD, from the best model's training run
CHECKAMG_DENOVO_DB_DATE = f"{CHECKAMG_DENOVO_DB_DATE[:4]}-{CHECKAMG_DENOVO_DB_DATE[4:6]}-{CHECKAMG_DENOVO_DB_DATE[6:]}"
print(f"De-novo DB version: {CHECKAMG_DENOVO_DB_VERSION} ({CHECKAMG_DENOVO_DB_DATE})")

De-novo DB version: 1.1 (2026-07-14)


In [5]:
dest = Path(
    f"./CheckAMG_denovo_db_v{str(CHECKAMG_DENOVO_DB_VERSION).replace('.0', '')}"
    f"_{CHECKAMG_DENOVO_DB_DATE.replace('-', '')}"
)
print(f"Output destination: {dest}")
os.makedirs(dest, exist_ok=True)

Output destination: CheckAMG_denovo_db_v1.1_20260714


In [6]:
SRC_CKPT_PATH = BEST_MODEL_CKPT
SRC_EMBED_PATH = BEST_MODEL_TRAIN_EMBED
SRC_INDEX_PATH = BEST_MODEL_TRAIN_EMBED.with_suffix(".index.faiss")
SRC_LABELS_PATH = BEST_MODEL_TRAIN_EMBED.with_suffix(".labels.h5")

In [7]:
OUT_PREFIX = f"{BEST_MODEL}.{runid_to_date_str(RUN_ID)}"

OUT_CKPT_PATH = dest.joinpath(f"{OUT_PREFIX}.ckpt")
OUT_EMBED_PATH = dest.joinpath(f"{OUT_PREFIX}.PST-EMBED.h5")
OUT_INDEX_PATH = dest.joinpath(f"{OUT_PREFIX}.PST-EMBED.index.faiss")
OUT_LABELS_PATH = dest.joinpath(f"{OUT_PREFIX}.PST-EMBED.labels.h5")
OUT_PREFIX

'checkAMG-PST_TL-P__large_5.20260714'

In [8]:
for src, dst in [
    (SRC_CKPT_PATH, OUT_CKPT_PATH),
    (SRC_EMBED_PATH, OUT_EMBED_PATH),
    (SRC_INDEX_PATH, OUT_INDEX_PATH),
    (SRC_LABELS_PATH, OUT_LABELS_PATH),
]:
    print(f"Copying {src} to {dst}...")
    shutil.copy(src, dst)
    assert dst.exists(), f"Failed to copy {src} to {dst}"
    assert dst.stat().st_size == src.stat().st_size, f"File size mismatch after copying {src} to {dst}"

Copying /storage2/scratch/kosmopoulos/projects/checkAMG/pst/training/train_outputs_chtc/checkamg_train_071426/checkAMG-PST_TL-P__large_5/lightning_logs/version_0/checkpoints/epoch=13_train_loss=0.114.ckpt to CheckAMG_denovo_db_v1.1_20260714/checkAMG-PST_TL-P__large_5.20260714.ckpt...
Copying /storage2/scratch/kosmopoulos/projects/checkAMG/pst/training/train_outputs_chtc/checkamg_train_071426/checkAMG-PST_TL-P__large_5/checkAMG_train_esm2_t30_150M.graphfmt.PST-EMBED.h5 to CheckAMG_denovo_db_v1.1_20260714/checkAMG-PST_TL-P__large_5.20260714.PST-EMBED.h5...
Copying /storage2/scratch/kosmopoulos/projects/checkAMG/pst/training/train_outputs_chtc/checkamg_train_071426/checkAMG-PST_TL-P__large_5/checkAMG_train_esm2_t30_150M.graphfmt.PST-EMBED.index.faiss to CheckAMG_denovo_db_v1.1_20260714/checkAMG-PST_TL-P__large_5.20260714.PST-EMBED.index.faiss...
Copying /storage2/scratch/kosmopoulos/projects/checkAMG/pst/training/train_outputs_chtc/checkamg_train_071426/checkAMG-PST_TL-P__large_5/checkAMG

In [9]:
readme = build_denovo_readme(
    CHECKAMG_DENOVO_DB_VERSION,
    CHECKAMG_DENOVO_DB_DATE,
    OUT_PREFIX,
)
print(readme)
with open(dest / "README.txt", "w") as f:
    f.write(readme)

CheckAMG de-novo database version 1.1 (2026-07-14)

This is the reference database used by 'checkamg denovo'. It contains a trained Protein Set
Transformer (PST) model and the precomputed reference protein embeddings, FAISS index, and
labels derived from the training data. 'checkamg denovo' embeds query proteins with the model
and assigns metabolic labels by distance-weighted k-nearest-neighbor search against the index.

If you use CheckAMG de-novo for your research, please also cite the Protein Set Transformer:
Martin et al. 2025 Nat. Commun. https://doi.org/10.1038/s41467-025-66049-4

Contents:
  checkAMG-PST_TL-P__large_5.20260714.ckpt                   trained CheckAMG-PST model checkpoint
  checkAMG-PST_TL-P__large_5.20260714.PST-EMBED.h5           reference protein PST embeddings (with labels)
  checkAMG-PST_TL-P__large_5.20260714.PST-EMBED.index.faiss  FAISS index over the reference embeddings
  checkAMG-PST_TL-P__large_5.20260714.PST-EMBED.labels.h5    reference protein labels


In [10]:
packaged = make_tar_gz_parallel(dest, threads=32)
size = packaged.stat().st_size / (1024 * 1024 * 1024)
print(f"Wrote: {packaged} ({size:.2f} GB)")

Wrote: /storage2/scratch/kosmopoulos/software/CheckAMG/notebooks/CheckAMG_denovo_db_v1.1_20260714.tar.gz (72.77 GB)
